# 04 — CatBoost candidate

This notebook fits the second cancellation-model candidate under exactly the same population, feature, and validation-queue rules as logistic regression. CatBoost is included because it can learn nonlinear patterns and interactions while handling the categorical booking fields directly.

The comparison is deliberately small: three tree depths with every other setting fixed. Validation selects the CatBoost configuration; holdout is not selected, scored, or evaluated.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "hotel_bookings.csv").exists()
)
sys.path.insert(0, str(PROJECT_ROOT / "modeling"))

import evaluation_policy as evaluation
from models import (
    CATEGORICAL_FEATURES, NUMERIC_FEATURES, as_catboost_features,
    make_catboost_model,
)
from prepare_data import (
    RAW_DATA_PATH, complete_180_rows,
    model_features, prepare_data,
)

pd.set_option("display.max_columns", 60)

## 1. Recreate the same fitting and validation populations

The model is fitted on the same 43,264 training-period rows whose outcomes were known before validation began. It scores the same 11,050 validation bookings used for logistic regression. Using the identical rows and features makes the later model comparison meaningful.

Holdout rows are deliberately not selected or scored in this notebook.

In [2]:
raw = pd.read_csv(RAW_DATA_PATH)
prepared = prepare_data(raw)
modeling_population = complete_180_rows(prepared)

validation_population = modeling_population.loc[
    modeling_population["booking_period_split"].eq("validation")
].copy()
validation_start = validation_population["booking_week_start"].min()

nominal_train = modeling_population.loc[
    modeling_population["booking_period_split"].eq("train")
].copy()
model_fit_population = nominal_train.loc[
    nominal_train["arrival_date"].lt(validation_start)
].copy()

X_fit = model_features(model_fit_population)
y_fit = model_fit_population["is_canceled"].copy()
X_validation = model_features(validation_population)
y_validation = validation_population["is_canceled"].copy()

population_summary = pd.DataFrame(
    {
        "rows": [len(model_fit_population), len(validation_population)],
        "booking weeks": [
            model_fit_population["booking_week_start"].nunique(),
            validation_population["booking_week_start"].nunique(),
        ],
        "cancellation rate": [y_fit.mean(), y_validation.mean()],
    },
    index=["model fitting", "validation"],
)
population_summary.style.format({"cancellation rate": "{:.2%}"})

,rows,booking weeks,cancellation rate
model fitting,43264,60,29.15%
validation,11050,13,31.76%


## 2. Use native categories and a small depth sensitivity

The reusable CatBoost definition and category conversion live in `modeling/models.py`; this notebook retains the fitting, sensitivity comparison, and explanation. CatBoost receives the same 27 approved booking-time features. The 11 categorical fields are passed as labels rather than one-hot encoded, and numeric features are not standardized because tree splits do not depend on their units. The prepared features contain no missing values, so no additional imputation is added.

Only tree depth is varied: 4, 6, and 8. Every candidate uses 500 iterations, a 0.05 learning rate, Logloss, and the same random seed. There is no class weighting or early stopping. Each candidate is fitted on `X_fit` and evaluated through the same validation risk-only queue.

The predeclared choice is the highest mean weekly precision@150; an exact tie is resolved by lower Brier score and then shallower depth. This is a small sensitivity check, not an exhaustive search.

In [3]:
X_fit = as_catboost_features(X_fit)
X_validation = as_catboost_features(X_validation)

display(
    pd.Series(
        {
            "numeric features": len(NUMERIC_FEATURES),
            "categorical features": len(CATEGORICAL_FEATURES),
            "total features": (
                len(NUMERIC_FEATURES) + len(CATEGORICAL_FEATURES)
            ),
        },
        name="count",
    ).to_frame()
)
print(
    "Missing feature values — fitting:",
    int(X_fit.isna().sum().sum()),
    "| validation:",
    int(X_validation.isna().sum().sum()),
)

,count
numeric features,16
categorical features,11
total features,27


Missing feature values — fitting: 0 | validation: 0


In [4]:
candidate_models = {}
candidate_probabilities = {}
sensitivity_rows = []

for depth in [4, 6, 8]:
    candidate_model = make_catboost_model(depth)
    candidate_model.fit(
        X_fit, y_fit, cat_features=CATEGORICAL_FEATURES
    )
    p_cancel = candidate_model.predict_proba(X_validation)[:, 1]

    candidate_scored = validation_population.copy()
    candidate_scored["p_cancel"] = p_cancel
    _, _, candidate_queue_summary, _ = (
        evaluation.simulate_weekly_policy(
            candidate_scored,
            "validation",
            policy="risk_only",
        )
    )
    candidate_metrics = evaluation.model_metric_summary(
        y_validation, p_cancel
    )

    candidate_models[depth] = candidate_model
    candidate_probabilities[depth] = p_cancel
    sensitivity_rows.append(
        {
            "depth": depth,
            "mean weekly precision@150": candidate_queue_summary[
                "mean weekly queue precision at capacity"
            ],
            "average precision": candidate_metrics["average_precision"],
            "Brier score": candidate_metrics["brier_score"],
        }
    )

sensitivity_results = pd.DataFrame(sensitivity_rows)
selection_order = sensitivity_results.sort_values(
    ["mean weekly precision@150", "Brier score", "depth"],
    ascending=[False, True, True],
)
selected_depth = int(selection_order.iloc[0]["depth"])
catboost_model = candidate_models[selected_depth]

validation_scored = validation_population.copy()
validation_scored["p_cancel"] = candidate_probabilities[selected_depth]

display(
    sensitivity_results.style.format(
        {
            "mean weekly precision@150": "{:.2%}",
            "average precision": "{:.4f}",
            "Brier score": "{:.4f}",
        }
    )
)
print("Selected depth:", selected_depth)
display(
    validation_scored["p_cancel"]
    .agg(["min", "mean", "median", "max"])
    .rename("selected predicted cancellation probability")
    .to_frame()
)

,depth,mean weekly precision@150,average precision,Brier score
0,4,57.49%,0.7682,0.1585
1,6,58.31%,0.7656,0.1644
2,8,57.28%,0.7606,0.1660


Selected depth: 6


,selected predicted cancellation probability
min,0.000042
mean,0.184751
median,0.053644
max,0.999575


## 3. Check the selected validation probabilities

Average precision measures ranking across validation, while the Brier score checks probability error and is lower when probabilities are closer to outcomes. ROC AUC is included only as a familiar supporting measure. The constant baseline assigns every row the fitting-period cancellation rate.

These checks support interpretation. The weekly queue metric remains the primary model-selection measure.

In [5]:
constant_probability = pd.Series(y_fit.mean(), index=y_validation.index)
probability_metrics = pd.DataFrame(
    {
        "selected CatBoost": evaluation.model_metric_summary(
            y_validation, validation_scored["p_cancel"]
        ),
        "constant training-rate baseline": evaluation.model_metric_summary(
            y_validation, constant_probability
        ),
    }
).T
probability_metrics.round(4)

,average_precision,roc_auc,brier_score
selected CatBoost,0.7656,0.8653,0.1644
constant training-rate baseline,0.3176,0.5000,0.2174


### Calibration by probability group

The mean predicted probability and observed cancellation rate should be reasonably close within each group if the model is well calibrated. This is a diagnostic only: validation is not reused to fit a calibration model here.

In [6]:
validation_scored["probability_group"] = pd.qcut(
    validation_scored["p_cancel"], q=10, duplicates="drop"
)
calibration_table = (
    validation_scored.groupby("probability_group", observed=True)
    .agg(
        bookings=("source_row_id", "size"),
        mean_predicted_probability=("p_cancel", "mean"),
        observed_cancellation_rate=("is_canceled", "mean"),
    )
)
calibration_table.style.format(
    {
        "mean_predicted_probability": "{:.2%}",
        "observed_cancellation_rate": "{:.2%}",
    }
)

,bookings,mean_predicted_probability,observed_cancellation_rate
probability_group,,,
"(-0.0009577, 0.00235]",1105,0.12%,0.18%
"(0.00235, 0.00508]",1106,0.35%,0.90%
"(0.00508, 0.0113]",1104,0.78%,3.89%
"(0.0113, 0.0266]",1105,1.79%,15.29%
"(0.0266, 0.0536]",1105,3.94%,20.09%
"(0.0536, 0.102]",1105,7.51%,33.76%
"(0.102, 0.172]",1105,13.36%,40.90%
"(0.172, 0.308]",1105,23.04%,49.05%
"(0.308, 0.646]",1106,44.13%,62.30%


## 4. Evaluate the selected risk-only weekly queue

Each Monday, the active and uncalled validation queue is ranked only by the selected model's fixed booking-time `p_cancel`, with up to 150 calls. Value and urgency are excluded so this result can be compared directly with logistic regression.

The repeated-random baseline uses the same standing queue and capacity without model-based ordering.

In [7]:
(
    catboost_calls,
    catboost_weekly,
    catboost_queue_summary,
    catboost_final_queue,
) = evaluation.simulate_weekly_policy(
    validation_scored,
    "validation",
    policy="risk_only",
)

display(
    catboost_weekly[
        [
            "call_date",
            "eligible_queue",
            "calls",
            "cancellations_reached",
            "precision_at_capacity",
        ]
    ].style.format({"precision_at_capacity": "{:.2%}"})
)

queue_fields = [
    "total calls",
    "cancellations reached",
    "mean weekly queue precision at capacity",
    "weekly precision IQR",
    "mean capacity utilization",
    "bookings still waiting after final decision",
]
catboost_queue_summary.loc[queue_fields].to_frame("selected CatBoost")

,call_date,eligible_queue,calls,cancellations_reached,precision_at_capacity
0,2016-09-05 00:00:00,542,150,80,53.33%
1,2016-09-12 00:00:00,846,150,71,47.33%
2,2016-09-19 00:00:00,1184,150,90,60.00%
3,2016-09-26 00:00:00,1731,150,78,52.00%
4,2016-10-03 00:00:00,1955,150,66,44.00%
5,2016-10-10 00:00:00,2020,150,71,47.33%
6,2016-10-17 00:00:00,2326,150,87,58.00%
7,2016-10-24 00:00:00,2724,150,93,62.00%
8,2016-10-31 00:00:00,2750,150,72,48.00%
9,2016-11-07 00:00:00,2704,150,92,61.33%


,selected CatBoost
total calls,1950.000000
cancellations reached,1137.000000
mean weekly queue precision at capacity,0.583077
weekly precision IQR,0.140000
mean capacity utilization,1.000000
bookings still waiting after final decision,2624.000000


In [8]:
random_results = evaluation.repeated_random_baseline(
    validation_scored,
    "validation",
    repeats=100,
)
primary_metric = "mean weekly queue precision at capacity"

queue_comparison = pd.Series(
    {
        "selected CatBoost": catboost_queue_summary[primary_metric],
        "random mean": random_results[primary_metric].mean(),
        "random 5th percentile": random_results[primary_metric].quantile(0.05),
        "random 95th percentile": random_results[primary_metric].quantile(0.95),
    },
    name="mean weekly precision@150",
)
queue_comparison.map(lambda value: f"{value:.2%}").to_frame()

,mean weekly precision@150
selected CatBoost,58.31%
random mean,24.84%
random 5th percentile,23.47%
random 95th percentile,25.90%


## 5. Inspect feature importance

CatBoost importance shows which input fields contributed most to changes in the fitted predictions. It does not show whether a feature increases or decreases risk, and it does not establish causality. High importance for agent, company, or country can also reflect dataset-specific patterns that may not remain stable.

In [9]:
feature_importance = catboost_model.get_feature_importance(
    prettified=True
).rename(
    columns={"Feature Id": "feature", "Importances": "importance"}
)
feature_importance.head(12).round({"importance": 2})

,feature,importance
0,country,17.93
1,required_car_parking_spaces,14.20
2,deposit_type,10.35
3,agent,9.73
4,lead_time,7.07
5,previous_cancellations,5.81
6,total_of_special_requests,5.80
7,customer_type,5.59
8,adr,4.03
9,arrival_date_week_number,3.09


## 6. What this candidate shows and what comes next

Depth 4, 6, and 8 produce mean weekly precision@150 of 57.49%, 58.31%, and 57.28%. Under the selection rule stated before fitting, depth 6 is the CatBoost choice and reaches 1,137 cancellations in 1,950 calls. Depth 4 has the best average precision and Brier score, so depth 6 is not uniformly better; it wins specifically on the primary operational metric.

The selected model ranks risk well, with average precision 0.766, but its probabilities remain too low in most calibration groups. Their validation mean is 18.48% while 31.76% of validation bookings canceled. The scores are therefore useful for ordering, but should not yet be described as precise probabilities or expected-loss estimates.

For the cross-model decision, CatBoost reaches only five more cancellations than logistic regression: 1,137 versus 1,132, or 58.31% versus 58.05% precision@150. CatBoost has better average precision, while its selected configuration has a slightly worse Brier score and is harder to explain. Under notebook 02's predeclared rule that materially similar queue performance favors the simpler model, logistic regression remains the recommended model for the policy notebook.

The next notebook should use the selected model to compare the risk-only, risk-times-value, and risk-times-value-times-urgency policies on validation. Only then should the model and policy be frozen and evaluated once on holdout.